# Abstract classes

An abstract class describes what its subclasses must provide. It cannot be
created itself.

In [1]:
from abc import ABC, abstractmethod


class Notifier(ABC):
    @abstractmethod
    def send(self, message):
        """Deliver one message."""

    def send_all(self, messages):
        for message in messages:
            self.send(message)

In [2]:
Notifier()

TypeError: Can't instantiate abstract class Notifier without an implementation for abstract method 'send'

In [3]:
class PrintNotifier(Notifier):
    def send(self, message):
        print(f"[SMS] {message}")


PrintNotifier().send_all(["Doors open", "The show starts"])

[SMS] Doors open
[SMS] The show starts


In [4]:
class Forgetful(Notifier):
    def deliver(self, message):
        print(message)


Forgetful()

TypeError: Can't instantiate abstract class Forgetful without an implementation for abstract method 'send'

The error appears as soon as the object is created, not later in the middle of the festival.

## Festival: where the programme is kept

In [5]:
import json
from pathlib import Path


class ProgrammeStorage(ABC):
    @abstractmethod
    def load(self) -> list[dict]:
        """The saved shows."""

    @abstractmethod
    def save(self, shows: list[dict]) -> None:
        """Replace the saved shows."""


class MemoryStorage(ProgrammeStorage):
    def __init__(self):
        self._shows = []

    def load(self):
        return list(self._shows)

    def save(self, shows):
        self._shows = list(shows)


class JsonStorage(ProgrammeStorage):
    def __init__(self, path):
        self.path = Path(path)

    def load(self):
        return json.loads(self.path.read_text(encoding="utf-8"))

    def save(self, shows):
        self.path.parent.mkdir(exist_ok=True)
        self.path.write_text(json.dumps(shows, ensure_ascii=False), encoding="utf-8")


def add_show(storage: ProgrammeStorage, show: dict) -> None:
    shows = storage.load()
    shows.append(show)
    storage.save(shows)

In [6]:
memory = MemoryStorage()
add_show(memory, {"title": "Late Laughs"})
print(memory.load())

file_storage = JsonStorage("output/programme.json")
file_storage.save([])
add_show(file_storage, {"title": "Théâtre night"})
print(file_storage.path.read_text(encoding="utf-8"))

[{'title': 'Late Laughs'}]
[{"title": "Théâtre night"}]


`add_show` does not know where the shows are kept. A `SqlStorage` can be added
without changing it, and tests use `MemoryStorage`: fast, no file.

## A family of exceptions

In [7]:
class FestivalError(Exception):
    """Every error of the festival."""


class NotFoundError(FestivalError):
    """Unknown artist, venue or show."""


class ScheduleConflictError(FestivalError):
    """A venue or an artist is busy."""


for error in [NotFoundError("No venue 'nowhere'."), ScheduleConflictError("Le Kiosque is busy.")]:
    try:
        raise error
    except FestivalError as caught:
        print(type(caught).__name__, "-", caught)

NotFoundError - No venue 'nowhere'.
ScheduleConflictError - Le Kiosque is busy.


One `except FestivalError` catches the whole family; code that needs precision catches `ScheduleConflictError` only.